In [2]:
!pip install pwntools

  Using cached pwntools-4.15.0-py2.py3-none-any.whl.metadata (5.3 kB)
Using cached pwntools-4.15.0-py2.py3-none-any.whl (12.9 MB)


In [3]:
from pwn import *

io = remote('nc 34.170.146.252', 10790)

# 1. まず1回実行して、フラグの長さ（バイト数）を取得する
io.recvuntil(b"Press Enter to get the encrypted flag...")
io.sendline(b"")
io.recvuntil(b"Encrypted flag: ")
first_cipher_hex = io.recvline().strip().decode()
first_cipher = bytes.fromhex(first_cipher_hex)
flag_length = len(first_cipher)

print(f"[*] Flag length detected: {flag_length} bytes")

# 2. 各文字位置の候補を初期化 (0から255までの全ての可能性を持つ集合)
# candidates[i] は i番目の文字の候補リスト
candidates = [set(range(256)) for _ in range(flag_length)]

# 3. 十分な回数ループして暗号文を収集する
ITERATIONS = 3000

print(f"[*] Collecting {ITERATIONS} ciphertexts. This might take a moment...")
for i in range(ITERATIONS):
    if i % 500 == 0 and i > 0:
        print(f"[*] {i} iterations done...")

    io.recvuntil(b"Press Enter to get the encrypted flag...")
    io.sendline(b"")
    io.recvuntil(b"Encrypted flag: ")
    cipher_hex = io.recvline().strip().decode()
    cipher = bytes.fromhex(cipher_hex)

    # 取得した暗号文の各バイトについて、「元の平文ではあり得ない値」として候補から削除
    for j in range(flag_length):
        if cipher[j] in candidates[j]:
            candidates[j].remove(cipher[j])

io.close()

# 4. 結果の集計と表示
flag = bytearray()
for j in range(flag_length):
    if len(candidates[j]) == 1:
        # 最後に1つだけ残った値が正解
        flag.append(list(candidates[j])[0])
    else:
        print(f"[!] Warning: Byte {j} has {len(candidates[j])} candidates left. Try increasing ITERATIONS.")
        flag.append(ord('?'))

print("\n[+] Recovered Flag:")
print(flag.decode(errors='ignore'))

UnsupportedOperation: fileno

In [4]:
import socket

# pwntoolsの recvuntil の代わりになるヘルパー関数
def recvuntil(sock, suffix):
    data = b""
    while not data.endswith(suffix):
        chunk = sock.recv(1)
        if not chunk:
            raise ConnectionError("サーバーから切断されました")
        data += chunk
    return data

# 1. サーバーへ接続
HOST = '34.170.146.252'
PORT = 10790

print(f"[*] Connecting to {HOST}:{PORT}...")
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.connect((HOST, PORT))

# 最初の1回を実行してフラグの長さを取得する
recvuntil(s, b"Press Enter to get the encrypted flag...")
s.sendall(b"\n")  # エンターキー(改行)を送信
recvuntil(s, b"Encrypted flag: ")

# 次の改行まで読み取って16進数文字列を取り出す
first_cipher_hex = recvuntil(s, b"\n").strip().decode()
first_cipher = bytes.fromhex(first_cipher_hex)
flag_length = len(first_cipher)

print(f"[*] Flag length detected: {flag_length} bytes")

# 2. 各文字位置の候補を初期化 (0から255までの全ての可能性)
candidates = [set(range(256)) for _ in range(flag_length)]

# 3. 十分な回数ループして暗号文を収集する
ITERATIONS = 3000

print(f"[*] Collecting {ITERATIONS} ciphertexts. This might take a moment...")
for i in range(ITERATIONS):
    if i % 500 == 0 and i > 0:
        print(f"[*] {i} iterations done...")

    recvuntil(s, b"Press Enter to get the encrypted flag...")
    s.sendall(b"\n")
    recvuntil(s, b"Encrypted flag: ")

    cipher_hex = recvuntil(s, b"\n").strip().decode()
    cipher = bytes.fromhex(cipher_hex)

    # 取得した暗号文の各バイトについて、候補から削除
    for j in range(flag_length):
        if cipher[j] in candidates[j]:
            candidates[j].remove(cipher[j])

# 通信終了
s.close()

# 4. 結果の集計と表示
flag = bytearray()
for j in range(flag_length):
    if len(candidates[j]) == 1:
        flag.append(list(candidates[j])[0])
    else:
        print(f"[!] Warning: Byte {j} has {len(candidates[j])} candidates left. Try increasing ITERATIONS.")
        flag.append(ord('?'))

print("\n[+] Recovered Flag:")
print(flag.decode(errors='ignore'))

[*] Connecting to 34.170.146.252:10790...
[*] Flag length detected: 69 bytes
[*] Collecting 3000 ciphertexts. This might take a moment...
[*] 500 iterations done...
[*] 1000 iterations done...
[*] 1500 iterations done...
[*] 2000 iterations done...
[*] 2500 iterations done...

[+] Recovered Flag:
Alpaca{All_flowers_fade_yet_some_never_made_the_worlds_unfairly_made}
